In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
import warnings
import os

warnings.filterwarnings("ignore")

In [12]:
data_path = r"C:\Users\jordan.bignell\OneDrive - Correla Limited\Desktop\data\t_shirt_data\tshirt_dataset_updated.csv"
os.makedirs(os.path.dirname(data_path), exist_ok=True)  # Ensure folder exists

In [13]:
# List all possible T-shirt sizes
all_sizes = ['XS', 'S', 'M', 'L', 'XL', '2XL', '3XL', '4XL', '5XL', '6XL']

# Create encoder and fit
size_encoder = LabelEncoder()
size_encoder.fit(all_sizes)


LabelEncoder()

In [14]:
if os.path.exists("tshirt_dataset_updated.csv"):
    df = pd.read_csv("tshirt_dataset_updated.csv")
    # Ensure UserProvided column exists
    if 'UserProvided' not in df.columns:
        df['UserProvided'] = False
    # Encode Size
    df['Size_encoded'] = size_encoder.transform(df['Size'])
    print("Loaded existing dataset with user inputs.")
else:
    print("No existing dataset found. Using synthetic data.")


Loaded existing dataset with user inputs.


In [15]:
# -----------------------------
# 2. Encode labels and scale features
# -----------------------------
X = df[['Height_cm', 'Weight_kg']].values
y = df['Size_encoded'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [16]:
# -----------------------------
# 3. Train initial XGBoost model
# -----------------------------
xgb_model = XGBClassifier(
    objective='multi:softprob',
    num_class=len(size_encoder.classes_),
    eval_metric='mlogloss',
    use_label_encoder=False,
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model.fit(X_scaled, y)

cv_scores = cross_val_score(xgb_model, X_scaled, y, cv=5, scoring='accuracy')
print(f"Initial cross-validated accuracy: {cv_scores.mean():.3f}")

Initial cross-validated accuracy: 0.694


In [17]:
# -----------------------------
# 4. Prediction function
# -----------------------------
def predict_tshirt_size(height_cm, weight_kg):
    input_scaled = scaler.transform([[height_cm, weight_kg]])
    probas = xgb_model.predict_proba(input_scaled)[0]
    pred_index = np.argmax(probas)
    predicted_size = size_encoder.inverse_transform([pred_index])[0]
    confidence = probas[pred_index]
    return predicted_size, confidence


In [18]:
# -----------------------------
# 5. User input with optional retraining
# -----------------------------
def main():
    while True:
        try:
            height = float(input("Enter your height in cm (or '0' to exit): ").strip())
            if height == 0:
                break
            weight = float(input("Enter your weight in kg: ").strip())
        except ValueError:
            print("Invalid input. Please enter numeric values.")
            continue

        pred_size, confidence = predict_tshirt_size(height, weight)
        print(f"Predicted size: {pred_size} (confidence: {confidence*100:.1f}%)")

        # Ask user for actual size to improve model
        actual_size = input("Enter your actual T-shirt size (or press Enter if unknown): ").strip().upper()
        if actual_size in size_encoder.classes_:
            # Add new data with a flag
            new_row = pd.DataFrame([[height, weight, actual_size, True]], 
                                   columns=['Height_cm', 'Weight_kg', 'Size', 'UserProvided'])
            new_row['Size_encoded'] = size_encoder.transform(new_row['Size'])
            
            global df, X_scaled, y
            if 'UserProvided' not in df.columns:
                df['UserProvided'] = False  # flag existing synthetic data
            
            df = pd.concat([df, new_row], ignore_index=True)

            # Prepare features
            X = df[['Height_cm', 'Weight_kg']].values
            y = df['Size_encoded'].values
            X_scaled = scaler.fit_transform(X)  # rescale

            # Retrain model
            xgb_model.fit(X_scaled, y)
            print("Model updated with new user data!")

            # Show the most recent user inputs
            print("\n--- Last 5 User Inputs ---")
            print(df[df['UserProvided']].tail(5))
            print("--------------------------\n")
            # Retrain model
            xgb_model.fit(X_scaled, y)
            print("Model updated with new user data!")

            # Recheck accuracy
            cv_scores_updated = cross_val_score(xgb_model, X_scaled, y, cv=5, scoring='accuracy')
            print(f"Updated cross-validated accuracy: {cv_scores_updated.mean():.3f}")
            # Save the updated dataset to CSV
            df.to_csv(data_path, index=False)
            print(f"Dataset saved to '{data_path}'.")


if __name__ == "__main__":
    main()


Predicted size: L (confidence: 94.4%)
Model updated with new user data!

--- Last 5 User Inputs ---
    Height_cm  Weight_kg Size  Size_encoded  UserProvided
81      175.0       80.0    M             6          True
82      150.0       50.0   XS             9          True
83      183.0       85.0    L             5          True
84      165.0       72.0    S             7          True
85      185.0       90.0    L             5          True
--------------------------

Model updated with new user data!
Updated cross-validated accuracy: 0.618
Dataset saved to 'C:\Users\jordan.bignell\OneDrive - Correla Limited\Desktop\data\t_shirt_data\tshirt_dataset_updated.csv'.
Predicted size: XS (confidence: 89.9%)
Model updated with new user data!

--- Last 5 User Inputs ---
    Height_cm  Weight_kg Size  Size_encoded  UserProvided
82      150.0       50.0   XS             9          True
83      183.0       85.0    L             5          True
84      165.0       72.0    S             7        

In [19]:
print(df.tail(10))  # shows the last 10 entries


    Height_cm  Weight_kg Size  Size_encoded  UserProvided
77      170.0       70.0    S             7         False
78      185.0       90.0    L             5         False
79      185.0       92.0    L             5         False
80      180.0       85.0    L             5         False
81      175.0       80.0    M             6          True
82      150.0       50.0   XS             9          True
83      183.0       85.0    L             5          True
84      165.0       72.0    S             7          True
85      185.0       90.0    L             5          True
86      160.0       60.0   XS             9          True
